## 1. Importazioni / Imports

**In italiano:** Importiamo le librerie necessarie per il progetto. Usiamo `langchain_google_genai` per il modello linguistico, `DuckDuckGoSearchRun` come tool di ricerca esterna, `langgraph` per creare il nostro grafo di agenti e `transformers` con `torch` per l'inferenza del modello NLI.

**In English:** Import the necessary libraries for the project. We use `langchain_google_genai` for the language model, `DuckDuckGoSearchRun` as an external search tool, `langgraph` to create our agent graph, and `transformers` with `torch` for NLI inference.

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.graph import StateGraph, START, END
from typing import Annotated, TypedDict, Any
from IPython.display import Image, display
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os

c:\Users\aless\Progetti Universitari\Sii\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Definizione degli Stati / State Definition

**In italiano:** Gli stati che verranno realizzati per il seguente progetto saranno 3:
* **InputState**: Uno stato di input, per permettere all'utente di inserire solo la query iniziale.
* **HiddenState**: Uno stato nascosto, che memorizza la query di ricerca raffinata, i risultati di ricerca (le evidenze/premessa), l'etichetta NLI e la spiegazione generata.
* **OutputState**: Uno stato di output, che conterrà solamente la risposta finale da mostrare all'utente.

**In English:** The states that will be implemented for this project will be 3:
* **InputState**: An input state to allow the user to input only the initial query.
* **HiddenState**: A hidden state that stores the refined search query, the search results (evidence/premise), the NLI label, and the generated explanation.
* **OutputState**: An output state that holds only the final response generated.

In [2]:
class InputState(TypedDict):
    query: str

class HiddenState(TypedDict):
    query: str
    search_query: str
    researches: str
    nli_label: str
    motivation: str
    response: str

class OutputState(TypedDict):
    response: str

## 3. Inizializzazione dei Modelli / Models Initialization

**In italiano:** Inizializziamo il modello LLM (`gemini-3.1-flash-lite-preview`) e il modello di Natural Language Inference (NLI) fine-tuned (DeBERTa).

**In English:** We initialize the LLM model (`gemini-3.1-flash-lite-preview`) and the fine-tuned Natural Language Inference (NLI) model (DeBERTa).

In [ ]:
llm = ChatGoogleGenerativeAI(model='gemma-4-31b-it')

# Inizializza il modello NLI fine-tuned. 
# Sostituire model_path con il percorso del proprio modello fine-tuned (es. '../fine-tuned_model').
model_path = "../fever-nli-deberta" 
if not os.path.exists(model_path):
    print(f"Modello locale non trovato in {model_path}. Uso il modello base da HuggingFace come fallback.")
    model_path = "cross-encoder/nli-deberta-v3-large"

tokenizer = AutoTokenizer.from_pretrained(model_path)
nli_model = AutoModelForSequenceClassification.from_pretrained(model_path)
nli_model.eval()

# Imposta il device (GPU se disponibile, altrimenti CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
nli_model.to(device)
print(f"Modello NLI caricato su: {device}")

Modello locale non trovato in ../fine-tuned_model. Uso il modello base da HuggingFace come fallback.


## 4. Definizione dei Nodi (Architettura RAG-NLI) / Nodes Definition

**In italiano:** Definiamo i nodi del grafo.
- `refine_query_node`: L'LLM estrae dalla notizia utente i concetti chiave per la ricerca sul web.
- `search_node`: Cerca le informazioni sul web (che faranno da *Premessa* o evidenza).
- `nli_classification_node`: Il modello NLI confronta l'evidenza trovata (Premessa) con la notizia dell'utente (Ipotesi) per emettere un verdetto fattuale (SUPPORTS, REFUTES, NOT ENOUGH INFO).
- `generate_motivation_node`: L'LLM formula una risposta discorsiva per l'utente, basandosi sul risultato del modello NLI e allegando le fonti.

**In English:** Let's define the graph nodes.
- `refine_query_node`: The LLM extracts key concepts from the user's news for web search.
- `search_node`: Searches for information on the web (which acts as the *Premise* or evidence).
- `nli_classification_node`: The NLI model compares the found evidence (Premise) with the user's news (Hypothesis) to issue a factual verdict.
- `generate_motivation_node`: The LLM formulates a narrative response for the user, based on the NLI model's result and attaching sources.

In [ ]:
def refine_query_node(state: HiddenState):
    prompt = f"""
    Devi fare una ricerca su Google/DuckDuckGo per verificare la seguente notizia: "{state['query']}"
    Estrai solo le parole chiave essenziali o formula una query di ricerca ottimale (max 5-6 parole).
    Rispondi unicamente con la stringa di ricerca, senza formattazione o introduzioni.
    """
    res = llm.invoke(prompt)
    content = res.content
    if isinstance(content, list):
        content = "".join(
            part.get('text', '') if isinstance(part, dict) else str(part)
            for part in content
            if not isinstance(part, dict) or part.get('type') != 'thinking'
        )
    search_query = str(content).strip()
    print(f"[Refine Node] Query di ricerca generata: {search_query}")
    return {'search_query': search_query}

def search_node(state: HiddenState):
    search = DuckDuckGoSearchRun()
    results = search.invoke(state['search_query'])
    print(f"[Search Node] Risultati estratti: {results[:100]}...")
    return {'researches': results}

def nli_classification_node(state: HiddenState):
    premise = state['researches']
    hypothesis = state['query']
    
    # Tokenizziamo Premessa e Ipotesi insieme
    inputs = tokenizer(premise, hypothesis, padding="max_length", truncation=True, max_length=256, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = nli_model(**inputs)
        
    logits = outputs.logits
    predicted_class_id = logits.argmax().item()
    
    mapping = {
        0: "SUPPORTS",
        1: "NOT ENOUGH INFO",
        2: "REFUTES"
    }
    
    label = mapping[predicted_class_id]
    print(f"[NLI Node] Etichetta NLI generata: {label}")
    return {'nli_label': label}

def generate_motivation_node(state: HiddenState):
    final_prompt = f"""
    L'utente ha richiesto di verificare questa notizia: "{state['query']}"
    
    Abbiamo cercato sul web e trovato queste informazioni fattuali: "{state['researches']}"
    
    Il nostro modello di intelligenza artificiale NLI ha classificato la veridicità della notizia come: "{state['nli_label']}" 
    (dove SUPPORTS = confermata dalle fonti, NOT ENOUGH INFO = fonti insufficienti per dare una conferma/smentita, REFUTES = smentita apertamente dalle fonti).
    
    Scrivi una risposta esauriente e logica per l'utente in cui spieghi perché la notizia ha ottenuto questa classificazione, 
    utilizzando e citando discorsivamente le informazioni fattuali che abbiamo trovato. Sii professionale ma chiaro.
    """
    res = llm.invoke(final_prompt)
    
    # Aggiungiamo i raw link o lo snippet della ricerca alla fine
    content = res.content
    if isinstance(content, list):
        content = "".join(
            part.get('text', '') if isinstance(part, dict) else str(part)
            for part in content
            if not isinstance(part, dict) or part.get('type') != 'thinking'
        )
    final_response = str(content) + "\n\n---\n**Evidenza grezza recuperata dal web (DuckDuckGo):**\n" + state['researches']
    
    return {'response': final_response}

## 5. Compilazione del Grafo / Graph Compilation

**In italiano:** Assembliamo il workflow lineare: estrazione query -> ricerca -> classificazione NLI -> generazione risposta.

**In English:** Assemble the linear workflow: query extraction -> search -> NLI classification -> response generation.

In [ ]:
workflow = StateGraph(state_schema=HiddenState, input_schema=InputState, output_schema=OutputState)

workflow.add_node("refine_query", refine_query_node)
workflow.add_node("search_web", search_node)
workflow.add_node("nli_classification", nli_classification_node)
workflow.add_node("generate_motivation", generate_motivation_node)

workflow.add_edge(START, "refine_query")
workflow.add_edge("refine_query", "search_web")
workflow.add_edge("search_web", "nli_classification")
workflow.add_edge("nli_classification", "generate_motivation")
workflow.add_edge("generate_motivation", END)

app = workflow.compile()

In [ ]:
try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    print("Errore nella generazione dell'immagine. Verifica le dipendenze.")

## 6. Test e Invocazione / Test and Invocation

**In italiano:** Testiamo il nuovo grafo (RAG-NLI) passando una query.

**In English:** Let's test the new graph (RAG-NLI) by passing a query.

In [ ]:
input_data = {"query": "Addio Bastoni, netta posizione dell’Inter: annuncio di Romano"}
result = app.invoke(input_data)

print("\n========================== RISPOSTA FINALE ==========================\n")
print(result['response'])